# API 调用 OpenAI  
样例来源： https://github.com/openai/openai-cookbook 

## 1. API调用 + Prompt Engineering
----
源代码： https://github.com/openai/openai-cookbook/blob/main/examples/completions_usage_api.ipynb

数据源： https://tianchi.aliyun.com/dataset/56 

### 1. 多轮填词对话


In [ ]:
from openai import OpenAI

client = OpenAI()
 
messages = [
    {"role": "system", 
     "content": "你是一个数据分析助手，回答要专业、简洁"}
]

print("🤖 AI Chat 已启动（输入 exit 退出）")

while True:
    user_input = input("\n你：")

    if user_input.lower() == "exit":
        break
 
    messages.append({"role": "user", "content": user_input})

 
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages
    )

    reply = response.choices[0].message.content

    print("\nAI：", reply)

    messages.append({"role": "assistant", "content": reply})

🤖 AI Chat 已启动（输入 exit 退出）

AI： 您好！请问有什么数据分析方面的问题需要帮助吗？

AI： 今天是星期五。需要帮您做其他数据分析相关的事情吗？

AI： 您好！请问还有其他需要帮助的数据分析问题吗？

AI： 抱歉，我无法实时获取洛杉矶的当前温度。建议您使用天气应用或访问天气网站获取最新的气温信息。如果需要，我可以帮助您分析历史气温数据或预测模型。

AI： 您好！如果您有任何数据分析相关的问题，欢迎随时告诉我！


### 2. SQL 生成器

In [ ]:
from openai import OpenAI
client = OpenAI()

def generate_mysql(question):
    prompt = f"""
你是一个电商广告数据分析师，请根据问题生成SQL。

数据库结构：

raw_sample(user_id, adgroup_id, clk, time_stamp)
ad_feature(adgroup_id, cate_id, brand, price)

user_profile(user_id, age_level, gender, occupation)
user_behavior_log(user_id, btag, cate, time_stamp)

字段说明：
- clk: 是否点击（1=点击，0=未点击）
- time_stamp: 时间戳

要求：
- 使用标准SQL
- 只返回MySQL，不要解释

问题：
{question}
"""

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content


In [14]:
generate_mysql("计算最近30天整体CTR")

'```sql\nSELECT \n    ROUND(SUM(clk) / COUNT(*), 4) AS ctr\nFROM \n    raw_sample\nWHERE \n    time_stamp >= UNIX_TIMESTAMP(CURDATE() - INTERVAL 30 DAY);\n```'

In [15]:
generate_mysql("统计每个广告的CTR，并按CTR排序")

'```sql\nSELECT\n    adgroup_id,\n    SUM(clk) / COUNT(*) AS ctr\nFROM\n    raw_sample\nGROUP BY\n    adgroup_id\nORDER BY\n    ctr DESC;\n```'

## 数据分析AI助手（带工具调用）

### 3. 

## 3. 用 LangGraph： “自动分析 + SQL执行 + 报告生成 Agent”